# M28 Benchmark-Pack Execution API

This notebook is a thin interactive surface over the existing StratLake execution API. Artifacts, manifests, summaries, inventories, and named output paths are the source of truth; notebook cell state is only for inspection.

In [ ]:
from pathlib import Path

from src.execution import run_benchmark_pack

Use an explicit notebook output root. For repeated attempts, change the attempt suffix instead of writing into a broad shared root.

In [ ]:
config_path = "configs/benchmark_packs/m22_scale_repro.yml"
output_root = Path("artifacts/notebooks/m28_benchmark_pack_execution_api/attempt_001")

Run the benchmark pack through `src.execution`. The notebook does not recreate benchmark-pack batching, checkpointing, inventory, or manifest logic.

In [ ]:
result = run_benchmark_pack(
    config_path,
    output_root=output_root,
    stop_after_batches=1,
)

result.notebook_summary()

Inspect canonical artifacts through the `ExecutionResult` named output paths.

In [ ]:
summary = result.load_summary_json("summary_json")
manifest = result.load_manifest()
inventory = result.load_output_json("inventory_json")
benchmark_matrix_path = result.output_path("benchmark_matrix_csv", must_exist=True)

{
    "status": summary.get("status"),
    "run_id": result.run_id,
    "output_keys": result.output_keys(),
    "manifest_run_id": manifest.get("run_id"),
    "inventory_keys": sorted(inventory)[:5],
    "benchmark_matrix_path": benchmark_matrix_path.as_posix(),
}

If you intentionally reuse an output root, inspect `_RUNNING.json`, `_SUCCESS.json`, `_FAILED.json`, the manifest, checkpoint, summary, and inventory before deciding whether to resume or rerun.

In [ ]:
status_markers = {
    name: (output_root / name).exists()
    for name in ("_RUNNING.json", "_SUCCESS.json", "_FAILED.json")
}
status_markers